In [9]:
from itertools import product
from torch import optim
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import numpy as np
import albumentations as Albu
import pandas as pd
from torch.utils.data import DataLoader
from torch.utils.data.sampler import RandomSampler
from warmup_scheduler import GradualWarmupScheduler
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score
from tqdm import tqdm
import os
import sys
sys.path.append('../../../')
from utils.dataset import PandasDataset
from utils.models import EfficientNetApiA

In [10]:
# Fixed configuration
seed = 42
num_workers = 4
output_classes = 5
n_epochs = 10
patience = 5
batch_size = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

ROOT_DIR = '../../..'
data_dir = '../../../..'
images_dir = os.path.join(data_dir, 'tiles')

os.makedirs('logs', exist_ok=True)
print(f"Epochs: {n_epochs} | Patience: {patience}")

Using device: cuda
Epochs: 10 | Patience: 5


In [11]:
# Hyperparameter grid
learning_rates  = [3e-4, 1e-4, 3e-3]
fine_tune_vals  = [100, 150]
dropout_rates   = [0.4, 0.5, 0.6]
param_grid = [
    {
        "learning_rate": lr,
        "fine_tune":     ft,
        "dropout_rate":  dr,
    }
    for lr, ft, dr, in product(
        learning_rates, fine_tune_vals, dropout_rates
    )
]

print(f"Total combinations: {len(param_grid)}")

Total combinations: 18


In [12]:
df_train_ = pd.read_csv(f"{ROOT_DIR}/data/train_5fold.csv")

def remove_nonexistent_images(df, images_dir):
    paths = df['image_id'].apply(lambda x: os.path.join(images_dir, f"{x}.png"))
    mask  = [os.path.isfile(p) for p in paths]
    return df[mask]

df_train_.columns = df_train_.columns.str.strip()

train_idx = np.where(df_train_['fold'] != 3)[0]
valid_idx = np.where(df_train_['fold'] == 3)[0]

df_train = df_train_.loc[train_idx].reset_index(drop=True)
df_val   = df_train_.loc[valid_idx].reset_index(drop=True)

df_train = remove_nonexistent_images(df_train, images_dir)
df_val   = remove_nonexistent_images(df_val,   images_dir)

print(f"Train: {len(df_train)} | Val: {len(df_val)}")

Train: 7215 | Val: 1805


In [13]:
# Datasets (transforms fixed; dataloaders are rebuilt per batch_size in the loop)
train_transforms = Albu.Compose([
    Albu.Transpose(p=0.5),
    Albu.VerticalFlip(p=0.5),
    Albu.HorizontalFlip(p=0.5),
])

train_dataset = PandasDataset(images_dir, df_train, transforms=train_transforms, format="png")
valid_dataset = PandasDataset(images_dir, df_val,   transforms=None,             format="png")

print(f"Datasets ready: train={len(train_dataset)}, val={len(valid_dataset)}")

Datasets ready: train=7215, val=1805


In [14]:
def decode_ordinal_predictions(logits):
    return (torch.sigmoid(logits) > 0.5).sum(dim=1)

In [15]:
def train_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    losses = []
    bar_progress = tqdm(loader, desc="Training")

    for batch_data, batch_targets, _ in bar_progress:
        batch_data = batch_data.to(device)
        batch_targets = batch_targets.to(device)  # Already in ordinal format from PandasDataset

        optimizer.zero_grad()

        # Forward pass
        logits = model(batch_data)
        loss = loss_fn(logits, batch_targets)

        # Backward pass
        loss.backward()
        optimizer.step()

        # Track loss
        loss_np = loss.detach().cpu().numpy()
        losses.append(loss_np)
        # smooth_loss = sum(train_loss[-100:]) / min(len(train_loss), 100)

        bar_progress.set_postfix({'loss': f'{loss_np:.5f}'})
    return np.mean(losses)


def validate(model, loader, loss_fn, device):
    model.eval()
    all_preds, all_targets, losses = [], [], []

    with torch.no_grad():
        for data, targets, _ in tqdm(loader, desc="Validation", leave=False):
            data   = data.to(device)
            logits = model(data)
            loss   = loss_fn(logits, targets.to(device))

            preds  = logits.sigmoid().sum(dim=1).round().long().cpu()  # FIX
            labels = targets.sum(dim=1).long()

            all_preds.append(preds)
            all_targets.append(labels)
            losses.append(loss.item())

    preds   = torch.cat(all_preds).numpy()
    targets = torch.cat(all_targets).numpy()

    return {
        'val_loss':  np.mean(losses),
        'val_kappa': cohen_kappa_score(targets, preds, weights='quadratic'),
        'val_acc':   accuracy_score(targets, preds),
        'val_f1':    f1_score(targets, preds, average='macro', zero_division=0),
    }


print("Training/validation functions defined.")

Training/validation functions defined.


In [ ]:
results = []

for run_idx, params in enumerate(param_grid):
    lr = params['learning_rate']
    ft = params['fine_tune']
    dr = params['dropout_rate']

    print(f"\n{'='*70}")
    print(f"[{run_idx+1}/{len(param_grid)}] lr={lr} | fine_tune={ft} | dropout={dr}")
    print('='*70)

    # Reproducibility per run
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Dataloaders — validation uses sequential sampler (deterministic)
    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                              num_workers=num_workers, sampler=RandomSampler(train_dataset))
    valid_loader = DataLoader(valid_dataset, batch_size=batch_size,
                              num_workers=num_workers, shuffle=False)  # FIX 1

    # Model
    base_model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    model      = EfficientNetApiA(model=base_model, output_dimensions=output_classes,
                                  fine_tune=ft, dropout_rate=dr).to(device)

    # Loss, optimizer, scheduler
    loss_fn   = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max',  # FIX 2
                                                     patience=2, factor=0.5)

    best_kappa        = -1.0
    epochs_no_improve = 0
    best_metrics      = None  # FIX 3: use None sentinel

    for epoch in range(1, n_epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, loss_fn, device)
        metrics    = validate(model, valid_loader, loss_fn, device)

        kappa = metrics['val_kappa']
        print(f"  Epoch {epoch:02d} | train_loss={train_loss:.4f} | "
              f"val_kappa={kappa:.4f} | val_acc={metrics['val_acc']:.4f}")

        scheduler.step(kappa)  # FIX 2: step scheduler on validation kappa

        if kappa > best_kappa:
            best_kappa        = kappa
            best_metrics      = metrics
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"  Early stop at epoch {epoch}.")
            break

    print(f"  Best kappa: {best_kappa:.4f}")

    # FIX 3: guard against no valid best_metrics (e.g. n_epochs=0)
    if best_metrics is not None:
        results.append({**params, **{f"best_{k}": v for k, v in best_metrics.items()}})
    else:
        results.append({**params, 'best_val_kappa': None})

    # Free GPU memory
    del model, optimizer, loss_fn, scheduler  # FIX 4: also delete scheduler
    torch.cuda.empty_cache()

print("\nGrid search complete.")
df_results = (pd.DataFrame(results)
                .sort_values('best_val_kappa', ascending=False)
                .reset_index(drop=True))
df_results.to_csv('logs/gridsearch_results.csv', index=False)
print(df_results.to_string())


[1/18] lr=0.0003 | fine_tune=100 | dropout=0.4


Training: 100%|██████████| 3608/3608 [10:05<00:00,  5.96it/s, loss=0.26559]


  Epoch 01 | train_loss=0.4243 | val_kappa=0.7071 | val_acc=0.3319


Training: 100%|██████████| 3608/3608 [10:02<00:00,  5.99it/s, loss=0.19944]


  Epoch 02 | train_loss=0.3349 | val_kappa=0.7509 | val_acc=0.4432


...

Training: 100%|██████████| 3608/3608 [09:55<00:00,  6.06it/s, loss=0.29730]


  Epoch 03 | train_loss=0.3063 | val_kappa=0.7622 | val_acc=0.5158


Training: 100%|██████████| 3608/3608 [09:22<00:00,  6.42it/s, loss=0.22688]


  Epoch 04 | train_loss=0.2873 | val_kappa=0.7800 | val_acc=0.5385


Validation:  31%|███▏      | 284/903 [00:31<01:07,  9.23it/s]

In [ ]:
import matplotlib.pyplot as plt

# Top-10 results
top10 = df_results.head(10)
print("Top 10 configurations by validation kappa:")
display(top10[['learning_rate','fine_tune','dropout_rate','focal_gamma','batch_size',
               'best_val_kappa','best_val_acc','best_val_f1','best_val_loss']])

# Bar chart of top-10 kappas
fig, ax = plt.subplots(figsize=(12, 4))
labels = [f"lr={r.learning_rate}\nft={r.fine_tune}\ndr={r.dropout_rate}\ng={r.focal_gamma}\nbs={r.batch_size}"
          for _, r in top10.iterrows()]
ax.bar(range(len(top10)), top10['best_val_kappa'], color='steelblue')
ax.set_xticks(range(len(top10)))
ax.set_xticklabels(labels, fontsize=7)
ax.set_ylabel('Quadratic Weighted Kappa')
ax.set_title('Top-10 Hyperparameter Combinations')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('logs/gridsearch_top10.png', dpi=150)
plt.show()

print(f"\nBest config: {df_results.iloc[0].to_dict()}")